# FlowGraphFormer — 01_Preprocessing_local.ipynb

Builds reproducible train, validation, and held-out test artifacts from the CSV
files in `dataset/raw/`. The notebook cleans and encodes the data, performs
feature selection, creates a stratified split, balances training data only,
scales features, generates dimensionality-reduction figures, and exports both
NumPy and Parquet/CSV formats under `dataset/processed/final/`.

Run `00A_Label_Converter.ipynb` only for numeric-label releases, then run
`00B_Dataset_Harmonizer.ipynb` when combining releases with different schemas.
Run this notebook top to bottom before either modeling notebook. Raw files are
read here; the two optional preparation notebooks are the only notebooks that
rewrite `dataset/raw/` (after making backups).


In [ ]:
# ==============================================================================
# FlowGraphFormer
# 01_Preprocessing_local.ipynb
#
# Cell 0 : Environment Detection & Initialization (Colab + Local)
# ==============================================================================

import os
import sys
import gc
import random
import warnings
import subprocess
import importlib
from pathlib import Path

warnings.filterwarnings("ignore")

print("=" * 80)
print("FlowGraphFormer")
print("Research Notebook : 01_Preprocessing")
print("=" * 80)

# ------------------------------------------------------------------------------
# Detect Runtime (Colab vs Local)
# ------------------------------------------------------------------------------

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

print(f"Runtime Detected      : {'Google Colab' if IN_COLAB else 'Local'}")

# ------------------------------------------------------------------------------
# Mount Google Drive (Colab only)
# ------------------------------------------------------------------------------

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

# ------------------------------------------------------------------------------
# Project Root
#   Colab : Google Drive project folder
#   Local : the folder the notebook is running from (or a sibling
#           "FlowGraphFormer" folder, if one exists)
# ------------------------------------------------------------------------------

PROJECT_NAME = "FlowGraphFormer"

if IN_COLAB:
    PROJECT_ROOT = Path("/content/drive/MyDrive/ML_Project") / PROJECT_NAME
else:
    _cwd = Path.cwd().resolve()
    if _cwd.name == PROJECT_NAME:
        PROJECT_ROOT = _cwd
    elif (_cwd / PROJECT_NAME).exists():
        PROJECT_ROOT = _cwd / PROJECT_NAME
    else:
        # Fall back to the current working directory as project root.
        # Edit this if your local project lives somewhere else, e.g.:
        # PROJECT_ROOT = Path.home() / "projects" / PROJECT_NAME
        PROJECT_ROOT = _cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project Root          : {PROJECT_ROOT}")

# ------------------------------------------------------------------------------
# Install Missing Packages
# ------------------------------------------------------------------------------

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "imblearn": "imbalanced-learn",
    "joblib": "joblib",
    "tqdm": "tqdm",
    "torch": "torch",
}

missing_packages = []

for module, package in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(module)
    except ImportError:
        missing_packages.append(package)

if missing_packages:
    print(f"\nInstalling missing packages: {missing_packages}\n")
    pip_cmd = [sys.executable, "-m", "pip", "install", "-q"]
    if not IN_COLAB:
        pip_cmd.append("--break-system-packages")
    subprocess.check_call(pip_cmd + missing_packages)
    print("Installation completed.\n")
else:
    print("\nAll required packages are already installed.\n")

# ------------------------------------------------------------------------------
# Random Seed
# ------------------------------------------------------------------------------

SEED = 42 #@param {type:"integer"}
random.seed(SEED)

import numpy as np
np.random.seed(SEED)

# ------------------------------------------------------------------------------
# PyTorch / Device Information
# ------------------------------------------------------------------------------

try:
    import torch

    if torch.cuda.is_available():
        torch.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)
        DEVICE = torch.device("cuda")
    elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        torch.manual_seed(SEED)
        DEVICE = torch.device("mps")
    else:
        torch.manual_seed(SEED)
        DEVICE = torch.device("cpu")

    print("-" * 80)
    print("PyTorch Information")
    print("-" * 80)
    print(f"Python Version  : {sys.version.split()[0]}")
    print(f"PyTorch Version : {torch.__version__}")
    print(f"CUDA Available  : {torch.cuda.is_available()}")

    if torch.cuda.is_available():
        print(f"GPU Name        : {torch.cuda.get_device_name(0)}")
        print(f"CUDA Version    : {torch.version.cuda}")
        print(
            "GPU Memory (GB) : "
            f"{round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)}"
        )
    else:
        print("GPU              : Not available (using CPU)")

    print(f"Device          : {DEVICE}")

except ImportError:
    DEVICE = "cpu"
    print("PyTorch is not available. Falling back to CPU-only mode.")

# ------------------------------------------------------------------------------
# NVIDIA GPU Status (only runs if nvidia-smi exists)
# ------------------------------------------------------------------------------

if subprocess.run(["bash", "-lc", "command -v nvidia-smi"], capture_output=True).returncode == 0:
    subprocess.run(["nvidia-smi"])
else:
    print("\nnvidia-smi not found (no NVIDIA GPU detected on this machine).")

# ------------------------------------------------------------------------------
# Clear GPU Cache
# ------------------------------------------------------------------------------

gc.collect()

try:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except NameError:
    pass

print("\n" + "=" * 80)
print("Random Seed            :", SEED)
print("Notebook Ready.")
print("=" * 80)


In [ ]:
# ==============================================================================
# Cell 2 : Imports
# ==============================================================================

# ------------------------------------------------------------------------------
# Standard Library
# ------------------------------------------------------------------------------

import os
import gc
import sys
import json
import time
import random
import warnings
from pathlib import Path
from datetime import datetime

# ------------------------------------------------------------------------------
# Scientific Computing
# ------------------------------------------------------------------------------

import numpy as np
import pandas as pd

# ------------------------------------------------------------------------------
# Visualization
# ------------------------------------------------------------------------------

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import seaborn as sns

# ------------------------------------------------------------------------------
# Progress Bar
# ------------------------------------------------------------------------------

from tqdm.auto import tqdm

# ------------------------------------------------------------------------------
# Scikit-Learn
# ------------------------------------------------------------------------------

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
)

from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
    LabelEncoder,
)

from sklearn.impute import SimpleImputer

from sklearn.decomposition import PCA

from sklearn.manifold import TSNE

from sklearn.ensemble import RandomForestClassifier

from sklearn.feature_selection import (
    mutual_info_classif,
    f_classif,
    SelectKBest,
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

# ------------------------------------------------------------------------------
# Imbalanced Learning
# ------------------------------------------------------------------------------

from imblearn.over_sampling import SMOTE

# ------------------------------------------------------------------------------
# PyTorch
# ------------------------------------------------------------------------------

import torch
import torch.nn as nn

# ------------------------------------------------------------------------------
# Utilities
# ------------------------------------------------------------------------------

import joblib

# ------------------------------------------------------------------------------
# Notebook Display
# ------------------------------------------------------------------------------

from IPython.display import (
    display,
    Markdown,
)

# ------------------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------------------

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", None)

sns.set_theme(
    style="whitegrid",
    context="talk",
)

plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.grid"] = True

# ------------------------------------------------------------------------------
# Memory Utilities
# ------------------------------------------------------------------------------
# Used throughout the notebook to keep RAM usage low. float64/int64 are
# rarely needed for NetFlow features -- downcasting to float32/int32
# (or smaller) typically cuts DataFrame memory usage in half or more,
# which is the difference between finishing on a Colab T4 runtime
# (~12-25GB RAM) and getting an "Out of Memory" crash.

def downcast_dtypes(df, verbose=True):
    """Downcast numeric columns in-place to the smallest safe dtype."""
    start_mb = df.memory_usage(deep=True).sum() / 1024 ** 2

    float_cols = df.select_dtypes(include=["float64"]).columns
    int_cols = df.select_dtypes(include=["int64"]).columns

    for col in float_cols:
        df[col] = pd.to_numeric(df[col], downcast="float")

    for col in int_cols:
        df[col] = pd.to_numeric(df[col], downcast="integer")

    end_mb = df.memory_usage(deep=True).sum() / 1024 ** 2

    if verbose:
        print(
            f"Downcast dtypes : {start_mb:,.1f} MB -> {end_mb:,.1f} MB "
            f"({(1 - end_mb / start_mb) * 100:.1f}% reduction)"
        )

    return df


def print_ram_usage(label=""):
    """Print current process RAM usage, if psutil is available."""
    try:
        import psutil

        process = psutil.Process(os.getpid())
        used_gb = process.memory_info().rss / 1024 ** 3

        try:
            total_gb = psutil.virtual_memory().total / 1024 ** 3
            print(f"RAM Usage{(' - ' + label) if label else ''} : {used_gb:.2f} GB / {total_gb:.2f} GB")
        except Exception:
            print(f"RAM Usage{(' - ' + label) if label else ''} : {used_gb:.2f} GB")

    except ImportError:
        pass


def free_memory(*names, scope):
    """Delete variables by name from the given scope (e.g. globals()) and gc.collect()."""
    for name in names:
        if name in scope:
            del scope[name]
    gc.collect()


print("=" * 80)
print("All Libraries Imported Successfully")
print("=" * 80)

In [ ]:
# ==============================================================================
# Cell 2 : Project Configuration
# ==============================================================================

# NOTE: IN_COLAB / PROJECT_ROOT were already set up in Cell 0 above.
# This cell just builds the dataset/report/model directory layout under
# whatever PROJECT_ROOT was resolved to (Drive path on Colab, local
# folder otherwise) -- no need to mount Drive or hardcode paths again.

# ==============================================================================
# Dataset Directories
# ==============================================================================

DATASET_DIR = PROJECT_ROOT / "dataset"

RAW_DATASET_DIR = DATASET_DIR / "raw"

PROCESSED_DATASET_DIR = DATASET_DIR / "processed"

# ==============================================================================
# Reports
# ==============================================================================

REPORT_DIR = PROJECT_ROOT / "reports"

PREPROCESS_REPORT_DIR = REPORT_DIR / "preprocessing"

FIGURE_DIR = PREPROCESS_REPORT_DIR / "figures"

TABLE_DIR = PREPROCESS_REPORT_DIR / "tables"

# ==============================================================================
# Saved Objects
# ==============================================================================

MODEL_DIR = PROJECT_ROOT / "models"

SCALER_DIR = MODEL_DIR / "scalers"

ENCODER_DIR = MODEL_DIR / "encoders"

FEATURE_DIR = MODEL_DIR / "features"

# ==============================================================================
# Create Directories
# ==============================================================================

DIRECTORIES = [

    PROJECT_ROOT,

    DATASET_DIR,

    RAW_DATASET_DIR,

    PROCESSED_DATASET_DIR,

    REPORT_DIR,

    PREPROCESS_REPORT_DIR,

    FIGURE_DIR,

    TABLE_DIR,

    MODEL_DIR,

    SCALER_DIR,

    ENCODER_DIR,

    FEATURE_DIR,

]

for directory in DIRECTORIES:

    directory.mkdir(

        parents=True,

        exist_ok=True,

    )

# ==============================================================================
# Configuration
# ==============================================================================

test_size = 15     #@param {type:"integer"}
validation_size = 15     #@param {type:"integer"}
smote_target_per_class = 50000     #@param {type:"integer"}
rf_estimators = 300    #@param {type:"integer"}

# ==============================================================================
# Feature Selection Toggle
# ==============================================================================
# USE_FEATURE_SELECTION = True  -> keep only the top `top_features` columns
#                                   (ranked by Random Forest importance)
# USE_FEATURE_SELECTION = False -> keep ALL engineered features; `top_features`
#                                   is ignored (RF/MI/ANOVA scores are still
#                                   computed and saved for reference/plots,
#                                   nothing is dropped from X_selected).
USE_FEATURE_SELECTION = True    #@param {type:"boolean"}
top_features = 25    #@param {type:"integer"}

figure_dpi = 300    #@param {type:"integer"}
figure_format = "png"    #@param {type:"string"}

CONFIG = {

    "random_seed": SEED,

    "test_size": test_size/100,

    "validation_size": validation_size/100,

    "smote_random_state": SEED,

    # Cap on synthetic samples per class. Prevents SMOTE's default
    # "auto" behavior of upsampling every minority class all the way
    # up to the majority class count (which is what caused the
    # 1GB -> 4.5GB blowup). Classes already above this count are left
    # untouched; classes below it are oversampled up to this target.
    "smote_target_per_class": smote_target_per_class,

    "rf_estimators": rf_estimators,

    # Whether Cell 9 reduces X down to the top-K ranked features, or
    # keeps every engineered feature. See toggle note above.
    "use_feature_selection": USE_FEATURE_SELECTION,

    "top_features": top_features,

    "figure_dpi": figure_dpi,

    "figure_format": figure_format,

}

# ==============================================================================
# Display Configuration
# ==============================================================================

print("=" * 80)

print("Project Configuration")

print("=" * 80)

print(f"Runtime                : {'Google Colab' if IN_COLAB else 'Local'}")

print(f"Project Root           : {PROJECT_ROOT}")

print(f"Raw Dataset            : {RAW_DATASET_DIR}")

print(f"Processed Dataset      : {PROCESSED_DATASET_DIR}")

print(f"Reports                : {PREPROCESS_REPORT_DIR}")

print(f"Figures                : {FIGURE_DIR}")

print(f"Tables                 : {TABLE_DIR}")

print(f"Scaler Directory       : {SCALER_DIR}")

print(f"Encoder Directory      : {ENCODER_DIR}")

print(f"Feature Directory      : {FEATURE_DIR}")

print(f"SMOTE Target / Class   : {CONFIG['smote_target_per_class']:,}")

fs_label = f"Top {CONFIG['top_features']} features" if CONFIG["use_feature_selection"] else "ALL features (selection disabled)"

print(f"Feature Selection      : {fs_label}")

print("=" * 80)

print("Configuration Loaded Successfully.")

print("=" * 80)

print_ram_usage("baseline (after config)")


In [ ]:
# ==============================================================================
# Cell 3 : Load & Merge Dataset
# ==============================================================================

import time
from tqdm.auto import tqdm

print("=" * 80)
print("Loading CIC-IDS2017 Dataset")
print("=" * 80)

start_time = time.time()

# ==============================================================================
# Detect CSV Files
# ==============================================================================

csv_files = sorted(

    RAW_DATASET_DIR.glob("*.csv")

)

if len(csv_files) == 0:

    raise FileNotFoundError(

        f"No CSV files found in:\n{RAW_DATASET_DIR}"

    )

print(f"Found {len(csv_files)} CSV file(s).\n")

# ==============================================================================
# Load Dataset
# ==============================================================================
# Memory strategy: each file is downcast to float32/int32 IMMEDIATELY
# after loading, before it ever joins the `datasets` list. This avoids
# holding the (much larger) float64 version of every file in RAM at
# once -- the peak RAM here is roughly HALF of what it would be if we
# downcast only after concatenation.

datasets = []

loading_report = []

for file in tqdm(

    csv_files,

    desc="Loading CSV Files",

):

    try:

        df = pd.read_csv(

            file,

            low_memory=False,

        )

        # Normalize column names immediately
        df.columns = (

            df.columns
            .str.strip()
            .str.replace("/", "_")
            .str.replace(" ", "_")
        )

        # Downcast immediately, per file, before appending to the list
        downcast_dtypes(df, verbose=False)

        datasets.append(df)

        loading_report.append(

            {

                "File": file.name,

                "Rows": df.shape[0],

                "Columns": df.shape[1],

                "Status": "Loaded",

            }

        )

    except Exception as e:

        loading_report.append(

            {

                "File": file.name,

                "Rows": 0,

                "Columns": 0,

                "Status": f"Failed ({e})",

            }

        )

# ==============================================================================
# Loading Report
# ==============================================================================

loading_report = pd.DataFrame(

    loading_report

)

display(loading_report)

# Save report
loading_report.to_csv(

    TABLE_DIR / "loading_report.csv",

    index=False,

)

# ==============================================================================
# Merge Dataset
# ==============================================================================

print("\nMerging datasets...\n")

dataset = pd.concat(

    datasets,

    ignore_index=True,

)

# Free the per-file list immediately -- once concatenated, we don't
# need the individual DataFrames anymore, and they're just dead weight.
free_memory("datasets", "df", scope=globals())

# ==============================================================================
# Memory Usage
# ==============================================================================

memory_mb = (

    dataset.memory_usage(

        deep=True

    ).sum()

    / 1024 ** 2

)

# ==============================================================================
# Dataset Information
# ==============================================================================

print("=" * 80)

print("Merged Dataset Summary")

print("=" * 80)

print(f"Rows              : {dataset.shape[0]:,}")

print(f"Columns           : {dataset.shape[1]}")

print(f"Memory Usage (MB) : {memory_mb:.2f}")

print(f"Load Time (sec)   : {time.time() - start_time:.2f}")

print("=" * 80)

print_ram_usage("after merge")

# ==============================================================================
# Save Merged Dataset (Parquet, Snappy-Compressed)
# ==============================================================================
# NOTE: switched from .csv to .parquet. Writing a multi-GB DataFrame as
# CSV builds a large in-memory text buffer before flushing to disk,
# which is a common cause of an OOM crash right at this step. Parquet
# writes column-by-column and is far lighter on RAM, plus the file
# ends up ~10-30x smaller on disk.

merged_dataset_path = (
    PROCESSED_DATASET_DIR
    / "merged_dataset.parquet"
)

try:
    dataset.to_parquet(
        merged_dataset_path,
        engine="pyarrow",
        compression="snappy",
        index=False,
    )
    print(f"\nMerged dataset saved successfully via pyarrow:\n{merged_dataset_path}")
except Exception as pyarrow_error:
    print(f"\npyarrow export failed ({pyarrow_error}). Trying fastparquet fallback...")
    try:
        dataset.to_parquet(
            merged_dataset_path,
            engine="fastparquet",
            compression="snappy",
            index=False,
        )
        print(f"\nMerged dataset saved successfully via fastparquet:\n{merged_dataset_path}")
    except Exception as fastparquet_error:
        csv_fallback_path = PROCESSED_DATASET_DIR / "merged_dataset.csv.gz"
        print(f"fastparquet failed ({fastparquet_error}). Saving as compressed CSV...")
        dataset.to_csv(csv_fallback_path, index=False, compression="gzip")
        print(f"\nMerged dataset saved successfully as compressed CSV:\n{csv_fallback_path}")

# Free memory
gc.collect()

print_ram_usage("after cell 3")


In [ ]:
# ==============================================================================
# Cell 4 : Dataset Overview
# ==============================================================================

print("=" * 80)
print("Dataset Overview")
print("=" * 80)

# ------------------------------------------------------------------------------
# Shape
# ------------------------------------------------------------------------------

rows, cols = dataset.shape

# Cache shape now -- `dataset` gets freed later (Cell 7), but the
# final report (Cell 18) still needs these numbers.
DATASET_SHAPE = (rows, cols)

print(f"Rows                 : {rows:,}")
print(f"Columns              : {cols}")

# ------------------------------------------------------------------------------
# Memory Usage
# ------------------------------------------------------------------------------

memory_mb = dataset.memory_usage(

    deep=True

).sum() / (1024 ** 2)

print(f"Memory Usage (MB)    : {memory_mb:.2f}")

# ------------------------------------------------------------------------------
# Data Types
# ------------------------------------------------------------------------------

dtype_summary = pd.DataFrame({

    "Data Type": dataset.dtypes.astype(str)

})

dtype_count = dtype_summary["Data Type"].value_counts()

print("\nData Type Summary")

display(dtype_count)

# ------------------------------------------------------------------------------
# Column Information
# ------------------------------------------------------------------------------

column_info = pd.DataFrame({

    "Column": dataset.columns,

    "Data Type": dataset.dtypes.astype(str),

    "Missing Values": dataset.isnull().sum().values,

    "Missing (%)": (

        dataset.isnull().sum().values

        / len(dataset)

    ) * 100,

})

display(column_info.head())

# ------------------------------------------------------------------------------
# Save Column Information
# ------------------------------------------------------------------------------

column_info.to_csv(

    TABLE_DIR / "column_information.csv",

    index=False,

)

# ------------------------------------------------------------------------------
# Dataset Summary
# ------------------------------------------------------------------------------

summary = {

    "Rows": rows,

    "Columns": cols,

    "Memory_MB": round(memory_mb, 2),

    "Duplicate_Rows": int(dataset.duplicated().sum()),

    "Missing_Values": int(dataset.isnull().sum().sum()),

}

summary_df = pd.DataFrame(

    summary.items(),

    columns=[

        "Metric",

        "Value",

    ],

)

display(summary_df)

summary_df.to_csv(

    TABLE_DIR / "dataset_summary.csv",

    index=False,

)

print("=" * 80)
print("Dataset Overview Completed")
print("=" * 80)
gc.collect()
print_ram_usage("after dataset overview")


In [ ]:
# ==============================================================================
# Cell 5 : Data Quality Analysis
# ==============================================================================

print("=" * 80)
print("Data Quality Analysis")
print("=" * 80)

# ==============================================================================
# Missing Values
# ==============================================================================

missing_count = dataset.isnull().sum()

missing_percent = (

    missing_count

    / len(dataset)

) * 100

missing_df = pd.DataFrame({

    "Feature": dataset.columns,

    "Missing Values": missing_count.values,

    "Missing (%)": missing_percent.values,

})

missing_df = missing_df.sort_values(

    by="Missing Values",

    ascending=False,

)

display(missing_df.head(20))

missing_df.to_csv(

    TABLE_DIR / "missing_values.csv",

    index=False,

)

# ==============================================================================
# Duplicate Rows
# ==============================================================================

duplicate_rows = dataset.duplicated().sum()

print(f"\nDuplicate Rows : {duplicate_rows:,}")

# ==============================================================================
# Infinite Values
# ==============================================================================

numeric_dataset = dataset.select_dtypes(

    include=[np.number]

)

infinite_count = np.isinf(

    numeric_dataset

).sum()

infinite_df = pd.DataFrame({

    "Feature": numeric_dataset.columns,

    "Infinite Values": infinite_count.values,

})

infinite_df = infinite_df.sort_values(

    by="Infinite Values",

    ascending=False,

)

display(infinite_df.head(20))

infinite_df.to_csv(

    TABLE_DIR / "infinite_values.csv",

    index=False,

)

# ==============================================================================
# Constant Columns
# ==============================================================================

constant_columns = [

    col

    for col in dataset.columns

    if dataset[col].nunique(dropna=False) == 1

]

constant_df = pd.DataFrame({

    "Constant Columns": constant_columns

})

display(constant_df)

constant_df.to_csv(

    TABLE_DIR / "constant_columns.csv",

    index=False,

)

# ==============================================================================
# Unique Values
# ==============================================================================

unique_df = pd.DataFrame({

    "Feature": dataset.columns,

    "Unique Values": [

        dataset[col].nunique()

        for col in dataset.columns

    ]

})

unique_df = unique_df.sort_values(

    by="Unique Values",

    ascending=False,

)

unique_df.to_csv(

    TABLE_DIR / "unique_values.csv",

    index=False,

)

# ==============================================================================
# Summary
# ==============================================================================

quality_summary = pd.DataFrame({

    "Metric": [

        "Rows",

        "Columns",

        "Duplicate Rows",

        "Total Missing Values",

        "Total Infinite Values",

        "Constant Columns",

    ],

    "Value": [

        len(dataset),

        dataset.shape[1],

        duplicate_rows,

        int(missing_count.sum()),

        int(infinite_count.sum()),

        len(constant_columns),

    ]

})

display(quality_summary)

# numeric_dataset was a full copy of every numeric column purely for
# the infinite-value check above -- free it now, it is not needed again.
free_memory("numeric_dataset", scope=globals())

quality_summary.to_csv(

    TABLE_DIR / "data_quality_summary.csv",

    index=False,

)

# ==============================================================================
# Missing Value Plot
# ==============================================================================

top_missing = missing_df.head(20)

plt.figure(figsize=(12,6))

sns.barplot(

    data=top_missing,

    x="Missing Values",

    y="Feature",

)

plt.title("Top 20 Features with Missing Values")

plt.tight_layout()

plt.savefig(

    FIGURE_DIR / "missing_values.png",

)

plt.show()

# ==============================================================================
# Infinite Value Plot
# ==============================================================================

top_infinite = infinite_df.head(20)

plt.figure(figsize=(12,6))

sns.barplot(

    data=top_infinite,

    x="Infinite Values",

    y="Feature",

)

plt.title("Top 20 Features with Infinite Values")

plt.tight_layout()

plt.savefig(

    FIGURE_DIR / "infinite_values.png",

)

plt.show()

print("=" * 80)
print("Data Quality Analysis Completed")
print("=" * 80)
gc.collect()
print_ram_usage("after data quality analysis")


In [ ]:
# ==============================================================================
# Cell 6 : Data Cleaning
# ==============================================================================

print("=" * 80)
print("Data Cleaning")
print("=" * 80)

original_shape = dataset.shape

print(f"Original Shape : {original_shape}")

# ==============================================================================
# Replace Infinite Values
# ==============================================================================

print("\nReplacing \u00b1Infinity with NaN...")

dataset.replace(

    [np.inf, -np.inf],

    np.nan,

    inplace=True,

)

# ==============================================================================
# Remove Duplicate Rows
# ==============================================================================

duplicates_before = dataset.duplicated().sum()

dataset.drop_duplicates(

    inplace=True,

)

dataset.reset_index(

    drop=True,

    inplace=True,

)

print(f"Duplicate Rows Removed : {duplicates_before:,}")

gc.collect()

# ==============================================================================
# Remove Constant Columns
# ==============================================================================

constant_columns = [

    col

    for col in dataset.columns

    if dataset[col].nunique(dropna=False) <= 1

]

if len(constant_columns) > 0:

    dataset.drop(

        columns=constant_columns,

        inplace=True,

    )

print(f"Constant Columns Removed : {len(constant_columns)}")

# ==============================================================================
# Missing Values
# ==============================================================================
# NOTE: fillna is applied column-by-column (not on the whole numeric
# block at once) to avoid materializing a second full copy of the
# numeric subset in memory just to compute the median.

missing_before = dataset.isnull().sum().sum()

numeric_columns = dataset.select_dtypes(

    include=np.number

).columns

for column in numeric_columns:

    if dataset[column].isnull().any():

        # FIX: `dataset[column].fillna(..., inplace=True)` is a chained-
        # assignment no-op under pandas Copy-on-Write (pandas 2.x+ /
        # always-on in pandas 3.0) -- it fills a temporary copy, not
        # `dataset` itself, so NaNs silently survive this loop and later
        # crash NaN-intolerant steps (e.g. mutual_info_classif in Cell 9).
        # Reassigning the column back is required, not inplace=True.

        dataset[column] = dataset[column].fillna(

            dataset[column].median()

        )

categorical_columns = dataset.select_dtypes(

    exclude=np.number

).columns

for column in categorical_columns:

    mode = dataset[column].mode(dropna=True)

    if len(mode) > 0:

        # Same fix as above -- reassign instead of inplace=True on a
        # chained `dataset[column]` selection.

        dataset[column] = dataset[column].fillna(

            mode.iloc[0]

        )

missing_after = dataset.isnull().sum().sum()

print(f"Missing Values Before : {missing_before:,}")

print(f"Missing Values After  : {missing_after:,}")

# ==============================================================================
# Normalize Column Names
# ==============================================================================

dataset.columns = (

    dataset.columns

    .str.strip()

    .str.replace(" ", "_", regex=False)

    .str.replace("/", "_", regex=False)

    .str.replace("-", "_", regex=False)

    .str.replace("(", "", regex=False)

    .str.replace(")", "", regex=False)

)

# ==============================================================================
# Downcast Dtypes
# ==============================================================================
# Infinities became NaN and got filled above, which can upcast columns
# back to float64 -- downcast again now that cleaning is done.

downcast_dtypes(dataset)

gc.collect()

# ==============================================================================
# Final Shape
# ==============================================================================

print("\nCleaning Completed")

print(f"Final Shape : {dataset.shape}")

print(f"Rows Removed : {original_shape[0] - dataset.shape[0]:,}")

print(f"Columns Removed : {original_shape[1] - dataset.shape[1]}")

print_ram_usage("after cleaning")

# ==============================================================================
# Cleaning Report
# ==============================================================================

cleaning_report = pd.DataFrame({

    "Metric":[

        "Original Rows",

        "Original Columns",

        "Final Rows",

        "Final Columns",

        "Duplicate Rows Removed",

        "Constant Columns Removed",

        "Missing Values Before",

        "Missing Values After",

    ],

    "Value":[

        original_shape[0],

        original_shape[1],

        dataset.shape[0],

        dataset.shape[1],

        duplicates_before,

        len(constant_columns),

        int(missing_before),

        int(missing_after),

    ]

})

display(cleaning_report)

cleaning_report.to_csv(

    TABLE_DIR / "cleaning_report.csv",

    index=False,

)

# ==============================================================================
# Save Clean Dataset (Parquet, Snappy-Compressed)
# ==============================================================================

clean_dataset_path = (

    PROCESSED_DATASET_DIR

    / "clean_dataset.parquet"

)

dataset.to_parquet(

    clean_dataset_path,

    engine="pyarrow",

    compression="snappy",

    index=False,

)

print()

print(f"Clean dataset saved to:")

print(clean_dataset_path)

print("=" * 80)

print_ram_usage("after cell 6")


In [ ]:
# ==============================================================================
# Cell 7 : Feature & Target Preparation
# ==============================================================================

print("=" * 80)
print("Feature & Target Preparation")
print("=" * 80)

# ==============================================================================
# Detect Target Column
# ==============================================================================

possible_targets = [

    "Label",
    "label",
    "Class",
    "class",
    "Attack",
    "attack",
    "Target",
    "target",

]

target_column = None

for column in possible_targets:

    if column in dataset.columns:

        target_column = column

        break

if target_column is None:

    raise ValueError(

        "Target column not found."

    )

print(f"Target Column : {target_column}")

# ==============================================================================
# Features & Labels
# ==============================================================================
# NOTE: X = dataset.drop(...) already creates a new DataFrame -- we do
# NOT also do X.copy() anywhere below. We also drop `dataset` itself
# right after, since nothing past this cell needs the full merged/
# cleaned frame anymore. Keeping it alive was effectively a 3rd full
# copy of the data sitting in RAM for the rest of the notebook.

X = dataset.drop(

    columns=[target_column]

)

y = dataset[target_column].copy()

print(f"Feature Shape : {X.shape}")

print(f"Label Shape   : {y.shape}")

# Free the full merged/cleaned dataset now -- everything we need from
# it (X, y) has already been extracted.
free_memory("dataset", scope=globals())

# ==============================================================================
# Encode Labels
# ==============================================================================

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

y_encoded = y_encoded.astype(np.int32)

# ==============================================================================
# Label Mapping
# ==============================================================================

label_mapping = pd.DataFrame({

    "Class ID": range(len(label_encoder.classes_)),

    "Class Name": label_encoder.classes_,

})

display(label_mapping)

# Save Label Mapping
label_mapping.to_csv(

    TABLE_DIR / "label_mapping.csv",

    index=False,

)

joblib.dump(

    label_encoder,

    ENCODER_DIR / "label_encoder.pkl",

)

# ==============================================================================
# Class Distribution
# ==============================================================================

class_distribution = (

    pd.Series(y)

    .value_counts()

    .reset_index()

)

class_distribution.columns = [

    "Class",

    "Count",

]

display(class_distribution)

class_distribution.to_csv(

    TABLE_DIR / "class_distribution.csv",

    index=False,

)

# ==============================================================================
# Dataset Information
# ==============================================================================

dataset_info = pd.DataFrame({

    "Metric":[

        "Samples",

        "Features",

        "Classes",

    ],

    "Value":[

        len(X),

        X.shape[1],

        len(label_encoder.classes_),

    ]

})

display(dataset_info)

dataset_info.to_csv(

    TABLE_DIR / "dataset_information.csv",

    index=False,

)

print_ram_usage("after feature/target prep")

# ==============================================================================
# Save Encoded Labels (Parquet, Snappy-Compressed)
# ==============================================================================
# NOTE: we no longer build a full `encoded_dataset = X.copy()` in
# memory -- that was a redundant full-size copy of X sitting in RAM
# purely to attach one extra label column before saving. Instead we
# write X and the encoded label column to disk separately and rely on
# row order (both are still index-aligned at this point) to join them
# back if ever needed, which avoids the copy entirely.

X.to_parquet(

    PROCESSED_DATASET_DIR / "encoded_features.parquet",

    engine="pyarrow",

    compression="snappy",

    index=False,

)

pd.DataFrame({

    target_column: y_encoded

}).to_parquet(

    PROCESSED_DATASET_DIR / "encoded_labels_only.parquet",

    engine="pyarrow",

    compression="snappy",

    index=False,

)

gc.collect()

print()

print("Encoded features saved.")

print("Encoded labels saved.")

print("Label encoder saved.")

print("Label mapping saved.")

print("=" * 80)

print_ram_usage("after cell 7")


In [ ]:
# ==============================================================================
# Cell 8 : Feature Validation & Engineering
# ==============================================================================

print("=" * 80)
print("Feature Validation & Engineering")
print("=" * 80)

# ==============================================================================
# Remove Identifier Columns
# ==============================================================================

identifier_keywords = [

    "Flow_ID",
    "FlowID",
    "Timestamp",
    "Source_IP",
    "Destination_IP",
    "Src_IP",
    "Dst_IP",
    "Src_Port",
    "Dst_Port",
    "Protocol_Name",

]

removed_columns = []

for column in list(X.columns):

    if column in identifier_keywords:

        removed_columns.append(column)

        X.drop(

            columns=column,

            inplace=True,

        )

print(f"Identifier Columns Removed : {len(removed_columns)}")

if removed_columns:

    print()

    for col in removed_columns:

        print(f" - {col}")

# ==============================================================================
# Keep Only Numeric Features
# ==============================================================================
# NOTE: dropping non-numeric columns in-place instead of building a
# separate `numeric_features = X.select_dtypes(...).copy()` frame --
# that pattern held two full copies of X in memory at once.

non_numeric = X.select_dtypes(exclude=np.number).columns.tolist()

if len(non_numeric) > 0:

    X.drop(

        columns=non_numeric,

        inplace=True,

    )

print()

print(f"Numeric Features     : {X.shape[1]}")

print(f"Non-Numeric Removed  : {len(non_numeric)}")

if len(non_numeric) > 0:

    print()

    for col in non_numeric:

        print(f" - {col}")

# ==============================================================================
# Remove Zero Variance Features
# ==============================================================================

variance = X.var()

zero_variance = variance[variance == 0].index.tolist()

if len(zero_variance) > 0:

    X.drop(

        columns=zero_variance,

        inplace=True,

    )

print()

print(f"Zero Variance Features Removed : {len(zero_variance)}")

# ==============================================================================
# Downcast Dtypes
# ==============================================================================

downcast_dtypes(X)

gc.collect()

# ==============================================================================
# Check Remaining Missing Values
# ==============================================================================

missing_total = X.isnull().sum().sum()

print()

print(f"Remaining Missing Values : {missing_total:,}")

# ==============================================================================
# Feature Statistics
# ==============================================================================

feature_statistics = pd.DataFrame({

    "Feature": X.columns,

    "Mean": X.mean().values,

    "Std": X.std().values,

    "Min": X.min().values,

    "Max": X.max().values,

    "Variance": X.var().values,

})

display(feature_statistics.head())

feature_statistics.to_csv(

    TABLE_DIR / "feature_statistics.csv",

    index=False,

)

# ==============================================================================
# Feature Summary
# ==============================================================================

summary = pd.DataFrame({

    "Metric":[

        "Samples",

        "Features",

        "Numeric Features",

        "Classes",

        "Removed Identifier Columns",

        "Removed Zero Variance",

        "Remaining Missing",

    ],

    "Value":[

        len(X),

        X.shape[1],

        X.shape[1],

        len(np.unique(y_encoded)),

        len(removed_columns),

        len(zero_variance),

        int(missing_total),

    ]

})

display(summary)

summary.to_csv(

    TABLE_DIR / "feature_summary.csv",

    index=False,

)

print_ram_usage("after feature validation")

# ==============================================================================
# Save Feature Matrix (Parquet, Snappy-Compressed)
# ==============================================================================

X.to_parquet(

    PROCESSED_DATASET_DIR / "all_features.parquet",

    engine="pyarrow",

    compression="snappy",

    index=False,

)

pd.DataFrame({

    target_column: y_encoded

}).to_parquet(

    PROCESSED_DATASET_DIR / "encoded_labels.parquet",

    engine="pyarrow",

    compression="snappy",

    index=False,

)

gc.collect()

print()

print("Feature matrix saved.")

print("Encoded labels saved.")

print("=" * 80)

print_ram_usage("after cell 8")


In [ ]:
# ==============================================================================
# Cell 9 : Feature Selection
# ==============================================================================

print("=" * 80)
print("Feature Selection")
print("=" * 80)

from sklearn.feature_selection import mutual_info_classif, f_classif

TOP_K = CONFIG["top_features"]
USE_FEATURE_SELECTION = CONFIG["use_feature_selection"]

if USE_FEATURE_SELECTION:
    print(f"Feature selection ENABLED -- will keep top {TOP_K} features.\n")
else:
    print("Feature selection DISABLED -- keeping ALL features.")
    print("(RF / Mutual Information / ANOVA scores below are still computed")
    print(" and saved for reference, but no columns will be dropped.)\n")

# ==============================================================================
# Sample for Feature Selection
# ==============================================================================
# NOTE: This is the single biggest RAM/crash fix in the notebook.
# Fitting a 300-tree RandomForest AND mutual_info_classif AND f_classif
# on the FULL dataset (millions of rows) is extremely memory- and
# CPU-hungry -- RandomForest alone builds 300 trees each holding a
# bootstrap sample's worth of indices/structure, and mutual_info_classif
# is especially expensive at this scale. None of these three methods
# need the full dataset to produce a stable feature ranking -- a large
# stratified sample gives essentially the same ranking at a fraction
# of the cost.

FEATURE_SELECTION_SAMPLE_SIZE = min(150_000, len(X))

if FEATURE_SELECTION_SAMPLE_SIZE < len(X):

    print(f"Sampling {FEATURE_SELECTION_SAMPLE_SIZE:,} of {len(X):,} rows for feature selection...")

    X_fs, _, y_fs, _ = train_test_split(

        X,

        y_encoded,

        train_size=FEATURE_SELECTION_SAMPLE_SIZE,

        stratify=y_encoded,

        random_state=SEED,

    )

else:

    X_fs, y_fs = X, y_encoded

print_ram_usage("before feature selection")

# ==============================================================================
# Random Forest
# ==============================================================================

print("\nTraining Random Forest...")

rf = RandomForestClassifier(

    n_estimators=CONFIG["rf_estimators"],

    max_depth=20,

    random_state=SEED,

    # NOTE: n_jobs=-1 spawns one worker process PER CPU CORE, and each
    # worker (loky backend) gets its own COPY of X_fs/y_fs. On a Colab
    # standard runtime (~12GB RAM, 2-4 cores) this alone can multiply
    # peak RAM by 2-4x and crash the runtime. n_jobs=2 caps that
    # duplication while still giving a meaningful speedup over 1.
    n_jobs=2,

)

rf.fit(

    X_fs,

    y_fs,

)

rf_importance = pd.Series(

    rf.feature_importances_,

    index=X.columns,

)

del rf
gc.collect()

# ==============================================================================
# Mutual Information
# ==============================================================================

print("Calculating Mutual Information...")

mi_scores = mutual_info_classif(

    X_fs,

    y_fs,

    random_state=SEED,

)

mi_scores = pd.Series(

    mi_scores,

    index=X.columns,

)

# ==============================================================================
# ANOVA
# ==============================================================================

print("Calculating ANOVA F-score...")

anova_scores, _ = f_classif(

    X_fs,

    y_fs,

)

anova_scores = pd.Series(

    anova_scores,

    index=X.columns,

)

# Done with the feature-selection sample -- free it before continuing
free_memory("X_fs", "y_fs", scope=globals())

# ==============================================================================
# Combine Results
# ==============================================================================

feature_scores = pd.DataFrame({

    "Feature": X.columns,

    "RandomForest": rf_importance.values,

    "MutualInformation": mi_scores.values,

    "ANOVA": anova_scores.values,

})

feature_scores = feature_scores.sort_values(

    by="RandomForest",

    ascending=False,

)

display(feature_scores.head(20))

# ==============================================================================
# Save Complete Scores
# ==============================================================================

feature_scores.to_csv(

    TABLE_DIR / "feature_importance.csv",

    index=False,

)

# ==============================================================================
# Select Top Features (or keep all, depending on the toggle)
# ==============================================================================

if USE_FEATURE_SELECTION:

    selected_features = feature_scores.head(

        TOP_K

    )["Feature"].tolist()

    print()

    print(f"Top {TOP_K} Features Selected")

    for i, feature in enumerate(selected_features, 1):

        print(f"{i:02d}. {feature}")

else:

    selected_features = X.columns.tolist()

    print()

    print(f"All {len(selected_features)} Features Kept (feature selection disabled)")

# ==============================================================================
# Reduce Dataset
# ==============================================================================
# NOTE: X_selected = X[selected_features] already returns a new,
# smaller DataFrame -- no .copy() needed on top of that.

X_selected = X[selected_features]

X_selected.to_parquet(

    PROCESSED_DATASET_DIR / "selected_features.parquet",

    engine="pyarrow",

    compression="snappy",

    index=False,

)

# Cache the original (pre-selection) feature count -- Cell 10 needs
# it for the summary table, but X itself is about to be freed.
ORIGINAL_FEATURE_COUNT = X.shape[1]

# The full-width X (all features) is no longer needed once we have
# X_selected -- free it to make room for the modeling steps ahead.
free_memory("X", scope=globals())

gc.collect()

print_ram_usage("after feature selection")

# ==============================================================================
# Save Feature List
# ==============================================================================

pd.DataFrame({

    "Selected Features": selected_features

}).to_csv(

    TABLE_DIR / "selected_features_list.csv",

    index=False,

)

# ==============================================================================
# Feature Importance Plot
# ==============================================================================

plt.figure(figsize=(12,8))

sns.barplot(

    data=feature_scores.head(20),

    x="RandomForest",

    y="Feature",

)

plt.title(

    "Top 20 Feature Importance (Random Forest)",

    fontsize=16,

)

plt.tight_layout()

plt.savefig(

    FIGURE_DIR / "feature_importance.png",

)

plt.show()

print()

print(f"Selected Features : {X_selected.shape[1]}")

print("=" * 80)

print_ram_usage("after cell 9")


In [ ]:
# ==============================================================================
# Cell 10 : Feature Selection Visualization
# ==============================================================================

print("=" * 80)
print("Feature Selection Visualization")
print("=" * 80)

# ==============================================================================
# Top 20 Random Forest Features
# ==============================================================================

top20_rf = feature_scores.head(20)

plt.figure(figsize=(14, 8))

sns.barplot(

    data=top20_rf,

    x="RandomForest",

    y="Feature",

)

plt.title(

    "Top 20 Features (Random Forest Importance)",

    fontsize=18,

)

plt.xlabel("Importance")

plt.ylabel("Feature")

plt.tight_layout()

plt.savefig(

    FIGURE_DIR / "top20_random_forest.png",

    dpi=CONFIG['figure_dpi'],

    bbox_inches="tight",

)

plt.show()

# ==============================================================================
# Top 20 Mutual Information
# ==============================================================================

top20_mi = feature_scores.sort_values(

    "MutualInformation",

    ascending=False,

).head(20)

plt.figure(figsize=(14,8))

sns.barplot(

    data=top20_mi,

    x="MutualInformation",

    y="Feature",

)

plt.title(

    "Top 20 Features (Mutual Information)",

    fontsize=18,

)

plt.tight_layout()

plt.savefig(

    FIGURE_DIR / "top20_mutual_information.png",

    dpi=CONFIG['figure_dpi'],

    bbox_inches="tight",

)

plt.show()

# ==============================================================================
# Top 20 ANOVA
# ==============================================================================

top20_anova = feature_scores.sort_values(

    "ANOVA",

    ascending=False,

).head(20)

plt.figure(figsize=(14,8))

sns.barplot(

    data=top20_anova,

    x="ANOVA",

    y="Feature",

)

plt.title(

    "Top 20 Features (ANOVA F-score)",

    fontsize=18,

)

plt.tight_layout()

plt.savefig(

    FIGURE_DIR / "top20_anova.png",

    dpi=CONFIG['figure_dpi'],

    bbox_inches="tight",

)

plt.show()

# ==============================================================================
# Correlation Heatmap (Selected Features)
# ==============================================================================

plt.figure(figsize=(18,16))

correlation = X_selected.corr()

sns.heatmap(

    correlation,

    cmap="coolwarm",

    center=0,

    square=True,

)

plt.title(

    "Correlation Heatmap (Selected Features)",

    fontsize=18,

)

plt.tight_layout()

plt.savefig(

    FIGURE_DIR / "selected_feature_correlation.png",

    dpi=CONFIG['figure_dpi'],

    bbox_inches="tight",

)

plt.show()

# ==============================================================================
# Correlation Matrix
# ==============================================================================

correlation.to_csv(

    TABLE_DIR / "selected_feature_correlation.csv",

)

# ==============================================================================
# Feature Selection Summary
# ==============================================================================

summary = pd.DataFrame({

    "Metric":[

        "Original Features",

        "Selected Features",

        "Selection Method",

        "Random Forest Trees",

    ],

    "Value":[

        ORIGINAL_FEATURE_COUNT,

        X_selected.shape[1],

        "Random Forest",

        CONFIG["rf_estimators"],

    ]

})

display(summary)

summary.to_csv(

    TABLE_DIR / "feature_selection_summary.csv",

    index=False,

)

print()

free_memory("correlation", scope=globals())
gc.collect()
print_ram_usage("after feature selection visualization")

print("Feature Selection Figures Saved")

print("=" * 80)

In [ ]:
# ==============================================================================
# Cell 11 : Train / Validation / Test Split
# ==============================================================================

print("=" * 80)
print("Train / Validation / Test Split")
print("=" * 80)

# ==============================================================================
# Configuration
# ==============================================================================

TRAIN_RATIO = 0.8    # @param {"type":"slider","min":0,"max":1,"step":0.01}
VALIDATION_RATIO = 0.10    # @param {"type":"slider","min":0,"max":1,"step":0.01}
TEST_RATIO = 0.10    # @param {"type":"slider","min":0,"max":1,"step":0.01}

assert abs(TRAIN_RATIO + VALIDATION_RATIO + TEST_RATIO - 1.0) < 1e-6

# ==============================================================================
# First Split (Train / Temp)
# ==============================================================================

X_train, X_temp, y_train, y_temp = train_test_split(

    X_selected,

    y_encoded,

    train_size=TRAIN_RATIO,

    random_state=SEED,

    stratify=y_encoded,

)

# ==============================================================================
# Second Split (Validation / Test)
# ==============================================================================

X_val, X_test, y_val, y_test = train_test_split(

    X_temp,

    y_temp,

    test_size=0.50,

    random_state=SEED,

    stratify=y_temp,

)

# X_selected/X_temp/y_temp are fully represented by X_train/X_val/X_test
# now -- free them rather than keeping 3 extra full-size copies around.
free_memory("X_selected", "X_temp", "y_temp", scope=globals())

gc.collect()

# ==============================================================================
# Verify Split
# ==============================================================================

total_samples = len(X_train) + len(X_val) + len(X_test)

train_percent = len(X_train) / total_samples * 100
val_percent = len(X_val) / total_samples * 100
test_percent = len(X_test) / total_samples * 100

print(f"Training Samples   : {len(X_train):,} ({train_percent:.2f}%)")
print(f"Validation Samples : {len(X_val):,} ({val_percent:.2f}%)")
print(f"Testing Samples    : {len(X_test):,} ({test_percent:.2f}%)")

print()

print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")

print_ram_usage("after train/val/test split")

# ==============================================================================
# Class Distribution
# ==============================================================================

train_distribution = pd.Series(y_train).value_counts().sort_index()

validation_distribution = pd.Series(y_val).value_counts().sort_index()

test_distribution = pd.Series(y_test).value_counts().sort_index()

distribution = pd.DataFrame({

    "Class": label_encoder.inverse_transform(train_distribution.index),

    "Train": train_distribution.values,

    "Validation": validation_distribution.values,

    "Test": test_distribution.values,

})

display(distribution)

distribution.to_csv(

    TABLE_DIR / "split_class_distribution.csv",

    index=False,

)

# ==============================================================================
# Save Split Dataset (Parquet, Snappy-Compressed)
# ==============================================================================

split_datasets = {

    "X_train.parquet": X_train,

    "X_validation.parquet": X_val,

    "X_test.parquet": X_test,

}

for filename, df in split_datasets.items():

    df.to_parquet(

        PROCESSED_DATASET_DIR / filename,

        engine="pyarrow",

        compression="snappy",

        index=False,

    )

labels = {

    "y_train.csv": y_train,

    "y_validation.csv": y_val,

    "y_test.csv": y_test,

}

for filename, label in labels.items():

    pd.DataFrame({

        "Label": label

    }).to_csv(

        PROCESSED_DATASET_DIR / filename,

        index=False,

    )

gc.collect()

# ==============================================================================
# Split Summary
# ==============================================================================

summary = pd.DataFrame({

    "Dataset":[

        "Training",

        "Validation",

        "Testing",

    ],

    "Samples":[

        len(X_train),

        len(X_val),

        len(X_test),

    ],

    "Percentage":[

        round(train_percent,2),

        round(val_percent,2),

        round(test_percent,2),

    ],

})

display(summary)

summary.to_csv(

    TABLE_DIR / "dataset_split_summary.csv",

    index=False,

)

# ==============================================================================
# Final Sanity Check
# ==============================================================================

assert len(y_train) == len(X_train)
assert len(y_val) == len(X_val)
assert len(y_test) == len(X_test)

print()

print("=" * 80)
print("Dataset Split Completed Successfully")
print("=" * 80)

print_ram_usage("after cell 11")


In [ ]:
# ==============================================================================
# Cell 12 : Balance Classes (Undersample Above Target, SMOTE Below Target)
# ==============================================================================
# CONFIG["smote_target_per_class"] now controls EVERY class, not just the
# minority ones:
#   - count > target  ->  RandomUnderSampler cuts it DOWN to target
#     (this is what fixes BENIGN dominating the training set)
#   - count < target  ->  SMOTE synthesizes it UP to target (unchanged
#     from before)
#   - count == target ->  left alone
# The two operations touch disjoint sets of classes, so it doesn't matter
# which runs first -- undersampling first is done here purely to shrink
# the data before SMOTE's interpolation step, which keeps peak RAM lower.

print("=" * 80)
print("Class Balancing (Under + Over, Capped at Target)")
print("=" * 80)

from imblearn.under_sampling import RandomUnderSampler

# ==============================================================================
# Class Distribution Before Balancing
# ==============================================================================

before_smote = pd.Series(y_train).value_counts().sort_index()

before_df = pd.DataFrame({

    "Class": label_encoder.inverse_transform(before_smote.index),

    "Count": before_smote.values,

})

display(before_df)

before_df.to_csv(

    TABLE_DIR / "before_smote_distribution.csv",

    index=False,

)

# ==============================================================================
# Build Under-sample and Over-sample Strategies
# ==============================================================================

target_per_class = CONFIG["smote_target_per_class"]

class_counts = pd.Series(y_train).value_counts()

undersample_strategy = {

    class_label: target_per_class

    for class_label, count in class_counts.items()

    if count > target_per_class

}

oversample_strategy = {

    class_label: target_per_class

    for class_label, count in class_counts.items()

    if count < target_per_class

}

print(f"\nTarget per Class        : {target_per_class:,}")

print(f"Classes to Undersample  : {len(undersample_strategy)} / {len(class_counts)}")

print(f"Classes to Oversample   : {len(oversample_strategy)} / {len(class_counts)}")

print(f"Classes Already at Target: {len(class_counts) - len(undersample_strategy) - len(oversample_strategy)} / {len(class_counts)}")

if undersample_strategy:

    under_df = pd.DataFrame({

        "Class": label_encoder.inverse_transform(list(undersample_strategy.keys())),

        "Current Count": [class_counts[k] for k in undersample_strategy.keys()],

        "Target Count": list(undersample_strategy.values()),

    })

    print("\nUndersampling (majority classes, e.g. BENIGN):")

    display(under_df)

if oversample_strategy:

    over_df = pd.DataFrame({

        "Class": label_encoder.inverse_transform(list(oversample_strategy.keys())),

        "Current Count": [class_counts[k] for k in oversample_strategy.keys()],

        "Target Count": list(oversample_strategy.values()),

    })

    print("\nOversampling (minority classes):")

    display(over_df)

# ==============================================================================
# Dynamic k_neighbors Clamping (for SMOTE step only)
# ==============================================================================
# SMOTE requires k_neighbors < smallest class count being oversampled.
# Very small classes (e.g. Heartbleed with only a handful of samples)
# will crash SMOTE with the default k_neighbors=5, so we clamp it.

smallest_class_count = class_counts[list(oversample_strategy.keys())].min() if oversample_strategy else class_counts.min()

k_neighbors = max(1, min(5, smallest_class_count - 1))

print(f"\nSmallest Class Involved (SMOTE) : {smallest_class_count:,}")

print(f"k_neighbors (clamped)           : {k_neighbors}")

# ==============================================================================
# Downcast Before Balancing
# ==============================================================================
# SMOTE generates synthetic float rows via interpolation -- it will
# upcast to float64 internally regardless of input dtype, but starting
# from float32 keeps peak memory during fit_resample() roughly half
# of what it would be starting from float64.

downcast_dtypes(X_train)

print_ram_usage("before balancing")

# ==============================================================================
# Step 1 : Undersample Classes Above Target
# ==============================================================================

if undersample_strategy:

    print("\nApplying RandomUnderSampler...")

    undersampler = RandomUnderSampler(

        sampling_strategy=undersample_strategy,

        random_state=CONFIG["smote_random_state"],

    )

    X_under, y_under = undersampler.fit_resample(

        X_train,

        y_train,

    )

    joblib.dump(

        undersampler,

        MODEL_DIR / "undersampler.pkl",

    )

    print("Undersampling Completed.")

else:

    print("\nNo classes above target -- skipping undersampling.")

    undersampler = None

    X_under, y_under = X_train, y_train

print_ram_usage("after undersampling")

# ==============================================================================
# Step 2 : SMOTE Classes Below Target
# ==============================================================================

print("\nApplying SMOTE...")

if oversample_strategy:

    smote = SMOTE(

        sampling_strategy=oversample_strategy,

        random_state=CONFIG["smote_random_state"],

        k_neighbors=k_neighbors,

    )

    X_train_smote, y_train_smote = smote.fit_resample(

        X_under,

        y_under,

    )

else:

    print("No classes below target -- skipping SMOTE.")

    smote = None

    X_train_smote, y_train_smote = X_under.copy(), y_under.copy()

print("SMOTE Completed.")

# Free the intermediate undersampled frame -- fully superseded by
# X_train_smote now.
if "X_under" in globals() and X_under is not X_train:

    free_memory("X_under", "y_under", scope=globals())

# ==============================================================================
# Distribution After Balancing
# ==============================================================================

after_smote = pd.Series(

    y_train_smote

).value_counts().sort_index()

after_df = pd.DataFrame({

    "Class": label_encoder.inverse_transform(after_smote.index),

    "Count": after_smote.values,

})

display(after_df)

after_df.to_csv(

    TABLE_DIR / "after_smote_distribution.csv",

    index=False,

)

# ==============================================================================
# Save Balanced Dataset (Parquet, Snappy-Compressed)
# ==============================================================================
# Parquet + snappy compression gives ~30x smaller files on disk vs. CSV,
# which matters a lot once row counts get into the hundreds of thousands.

X_train_smote.to_parquet(

    PROCESSED_DATASET_DIR / "X_train_smote.parquet",

    engine="pyarrow",

    compression="snappy",

    index=False,

)

pd.DataFrame({

    "Label": y_train_smote

}).to_parquet(

    PROCESSED_DATASET_DIR / "y_train_smote.parquet",

    engine="pyarrow",

    compression="snappy",

    index=False,

)

if smote is not None:

    joblib.dump(

        smote,

        MODEL_DIR / "smote.pkl",

    )

# Downcast the output back to float32 (fit_resample upcasts internally).
# NOTE: we deliberately do NOT free X_train here -- Cell 13 still needs
# it for feature_names, and Cell 17 still needs it to export X_train.csv.
# It gets freed at the end of Cell 17, once everything that depends on
# it is done.
downcast_dtypes(X_train_smote)

gc.collect()

print_ram_usage("after balancing")

# ==============================================================================
# Before Balancing Plot
# ==============================================================================

plt.figure(figsize=(12,6))

sns.barplot(

    data=before_df,

    x="Class",

    y="Count",

)

plt.xticks(rotation=45)

plt.title(

    "Class Distribution Before Balancing",

    fontsize=16,

)

plt.tight_layout()

plt.savefig(

    FIGURE_DIR / "before_smote.png",

    dpi=300,

)

plt.show()

# ==============================================================================
# After Balancing Plot
# ==============================================================================

plt.figure(figsize=(12,6))

sns.barplot(

    data=after_df,

    x="Class",

    y="Count",

)

plt.xticks(rotation=45)

plt.title(

    "Class Distribution After Balancing (Under + Over, Capped)",

    fontsize=16,

)

plt.tight_layout()

plt.savefig(

    FIGURE_DIR / "after_smote.png",

    dpi=300,

)

plt.show()

# ==============================================================================
# Summary
# ==============================================================================

summary = pd.DataFrame({

    "Metric":[

        "Training Samples Before",

        "Training Samples After",

        "Number of Classes",

        "Target per Class",

        "Classes Undersampled",

        "Classes Oversampled (SMOTE)",

        "Classes Already at Target",

        "k_neighbors Used (SMOTE)",

    ],

    "Value":[

        len(y_train),

        len(y_train_smote),

        len(after_df),

        target_per_class,

        len(undersample_strategy),

        len(oversample_strategy),

        len(class_counts) - len(undersample_strategy) - len(oversample_strategy),

        k_neighbors,

    ]

})

display(summary)

summary.to_csv(

    TABLE_DIR / "smote_summary.csv",

    index=False,

)

print()

print("Balanced Training Dataset Saved Successfully (Parquet, Snappy).")

print("=" * 80)

print_ram_usage("after cell 12")


In [ ]:
# ==============================================================================
# Cell 13 : Feature Scaling
# ==============================================================================

print("=" * 80)
print("Feature Scaling")
print("=" * 80)

from sklearn.preprocessing import StandardScaler

# ==============================================================================
# Initialize Scaler
# ==============================================================================

scaler = StandardScaler()

# ==============================================================================
# Fit ONLY on Training Data (SMOTE)
# ==============================================================================

print("Fitting StandardScaler on Training Set...")

X_train_scaled = scaler.fit_transform(

    X_train_smote

)

# ==============================================================================
# Transform Validation & Test
# ==============================================================================

X_validation_scaled = scaler.transform(

    X_val

)

X_test_scaled = scaler.transform(

    X_test

)

# ==============================================================================
# Convert Back to DataFrame
# ==============================================================================

feature_names = X_train.columns.tolist()

X_train_scaled = pd.DataFrame(

    X_train_scaled,

    columns=feature_names,

)

X_validation_scaled = pd.DataFrame(

    X_validation_scaled,

    columns=feature_names,

)

X_test_scaled = pd.DataFrame(

    X_test_scaled,

    columns=feature_names,

)

# ==============================================================================
# Save Scaler
# ==============================================================================

joblib.dump(

    scaler,

    MODEL_DIR / "standard_scaler.pkl",

)

# ==============================================================================
# Save Scaled Dataset (Parquet, Snappy-Compressed)
# ==============================================================================
# NOTE: These are saved as .parquet, not .csv. If you have older code
# (e.g. in 03_Big_Modeling.ipynb) that reads "X_train_scaled.csv", update
# it to read "X_train_scaled.parquet" via pd.read_parquet(...) instead
# of pd.read_csv(...). Also remember: these files are ALREADY
# SMOTE-balanced, so USE_SMOTE_PER_FOLD should be set to False
# wherever this data is loaded downstream.

X_train_scaled.to_parquet(

    PROCESSED_DATASET_DIR / "X_train_scaled.parquet",

    engine="pyarrow",

    compression="snappy",

    index=False,

)

X_validation_scaled.to_parquet(

    PROCESSED_DATASET_DIR / "X_validation_scaled.parquet",

    engine="pyarrow",

    compression="snappy",

    index=False,

)

X_test_scaled.to_parquet(

    PROCESSED_DATASET_DIR / "X_test_scaled.parquet",

    engine="pyarrow",

    compression="snappy",

    index=False,

)

# ==============================================================================
# Scaling Verification
# ==============================================================================

verification = pd.DataFrame({

    "Statistic":[

        "Mean",

        "Std",

        "Minimum",

        "Maximum",

    ],

    "Value":[

        X_train_scaled.values.mean(),

        X_train_scaled.values.std(),

        X_train_scaled.values.min(),

        X_train_scaled.values.max(),

    ]

})

display(verification)

verification.to_csv(

    TABLE_DIR / "scaling_verification.csv",

    index=False,

)

# ==============================================================================
# Distribution Before Scaling
# ==============================================================================

sample_feature = feature_names[0]

plt.figure(figsize=(12,5))

sns.histplot(

    X_train_smote[sample_feature],

    kde=True,

)

plt.title(

    f"Before Scaling ({sample_feature})"

)

plt.tight_layout()

plt.savefig(

    FIGURE_DIR / "before_scaling.png",

    dpi=CONFIG['figure_dpi'],

)

plt.show()

# ==============================================================================
# Distribution After Scaling
# ==============================================================================

plt.figure(figsize=(12,5))

sns.histplot(

    X_train_scaled[sample_feature],

    kde=True,

)

plt.title(

    f"After Scaling ({sample_feature})"

)

plt.tight_layout()

plt.savefig(

    FIGURE_DIR / "after_scaling.png",

    dpi=CONFIG['figure_dpi'],

)

plt.show()

# ==============================================================================
# Summary
# ==============================================================================

summary = pd.DataFrame({

    "Metric":[

        "Training Samples",

        "Validation Samples",

        "Testing Samples",

        "Number of Features",

        "Scaling Method",

        "Output Format",

    ],

    "Value":[

        len(X_train_scaled),

        len(X_validation_scaled),

        len(X_test_scaled),

        X_train_scaled.shape[1],

        "StandardScaler",

        "Parquet (Snappy)",

    ]

})

display(summary)

summary.to_csv(

    TABLE_DIR / "scaling_summary.csv",

    index=False,

)

print()

print("Feature Scaling Completed Successfully.")

print("=" * 80)

gc.collect()
print_ram_usage("after feature scaling")


In [ ]:
# ==============================================================================
# Cell 14 : PCA Visualization
# ==============================================================================

print("=" * 80)
print("Principal Component Analysis (PCA)")
print("=" * 80)

from sklearn.decomposition import PCA

# ==============================================================================
# PCA
# ==============================================================================

print("Applying PCA...")

pca = PCA(

    n_components=2,

    random_state=SEED,

)

X_pca = pca.fit_transform(

    X_train_scaled

)

# ==============================================================================
# PCA DataFrame
# ==============================================================================

pca_df = pd.DataFrame({

    "PC1": X_pca[:, 0],

    "PC2": X_pca[:, 1],

    "Label": label_encoder.inverse_transform(y_train_smote),

})

# X_pca is fully represented by pca_df now -- free the raw array.
free_memory("X_pca", scope=globals())

# ==============================================================================
# Explained Variance
# ==============================================================================

explained_variance = pca.explained_variance_ratio_

variance_df = pd.DataFrame({

    "Principal Component": [

        "PC1",

        "PC2",

    ],

    "Explained Variance (%)": explained_variance * 100,

})

display(variance_df)

variance_df.to_csv(

    TABLE_DIR / "pca_variance.csv",

    index=False,

)

# ==============================================================================
# PCA Scatter Plot
# ==============================================================================

plt.figure(figsize=(12,8))

sns.scatterplot(

    data=pca_df,

    x="PC1",

    y="PC2",

    hue="Label",

    s=12,

    alpha=0.7,

)

plt.title(

    "PCA Visualization of Training Dataset",

    fontsize=18,

)

plt.xlabel(

    f"PC1 ({explained_variance[0]*100:.2f}%)"

)

plt.ylabel(

    f"PC2 ({explained_variance[1]*100:.2f}%)"

)

plt.legend(

    bbox_to_anchor=(1.02,1),

    loc="upper left",

)

plt.tight_layout()

plt.savefig(

    FIGURE_DIR / "pca_visualization.png",

    dpi=CONFIG['figure_dpi'],

)

plt.show()

# ==============================================================================
# Save PCA Dataset
# ==============================================================================

pca_df.to_csv(

    PROCESSED_DATASET_DIR / "pca_dataset.csv",

    index=False,

)

joblib.dump(

    pca,

    MODEL_DIR / "pca.pkl",

)

# ==============================================================================
# Summary
# ==============================================================================

summary = pd.DataFrame({

    "Metric":[

        "Samples",

        "Original Features",

        "Principal Components",

        "Variance Explained (%)",

    ],

    "Value":[

        len(X_train_scaled),

        X_train_scaled.shape[1],

        2,

        round(explained_variance.sum()*100,2),

    ]

})

display(summary)

summary.to_csv(

    TABLE_DIR / "pca_summary.csv",

    index=False,

)

print()

print("PCA Completed Successfully.")

print("=" * 80)
gc.collect()
print_ram_usage("after PCA")


In [ ]:
# ==============================================================================
# Cell 15 : t-SNE Visualization
# ==============================================================================

print("=" * 80)
print("t-SNE Visualization")
print("=" * 80)

from sklearn.manifold import TSNE

# ==============================================================================
# Configuration
# ==============================================================================

# True  -> use entire dataset
# False -> sample dataset
use_full_dataset = False        #@param {type:"boolean"}

# Number of samples if use_full_dataset=False
sample_size = 10000       #@param {type:"integer"}

# Recommended: True
stratified_sampling = True       #@param {type:"boolean"}

TSNE_CONFIG = {

    # True  -> use entire dataset
    # False -> sample dataset
    "use_full_dataset": use_full_dataset,

    # Number of samples if use_full_dataset=False
    "sample_size": sample_size,

    # Recommended: True
    "stratified_sampling": stratified_sampling,

    # t-SNE Parameters
    "perplexity": 30,
    "learning_rate": "auto",
    "init": "pca",
    "random_state": SEED,

}

# ==============================================================================
# Prepare Dataset
# ==============================================================================

if TSNE_CONFIG["use_full_dataset"]:

    print("\nUsing FULL dataset for t-SNE.")

    X_tsne_input = X_train_scaled.copy()

    y_tsne = y_train_smote.copy()

else:

    print("\nUsing SAMPLE dataset for t-SNE.")

    sample_size = min(

        TSNE_CONFIG["sample_size"],

        len(X_train_scaled)

    )

    if TSNE_CONFIG["stratified_sampling"]:

        _, X_tsne_input, _, y_tsne = train_test_split(

            X_train_scaled,

            y_train_smote,

            test_size=sample_size,

            stratify=y_train_smote,

            random_state=SEED,

        )

    else:

        indices = np.random.choice(

            len(X_train_scaled),

            sample_size,

            replace=False,

        )

        X_tsne_input = X_train_scaled.iloc[indices]

        y_tsne = y_train_smote[indices]

print(f"\nSamples used : {len(X_tsne_input):,}")

# ==============================================================================
# Run t-SNE
# ==============================================================================

print("\nRunning t-SNE...")

tsne = TSNE(

    n_components=2,

    perplexity=TSNE_CONFIG["perplexity"],

    learning_rate=TSNE_CONFIG["learning_rate"],

    init=TSNE_CONFIG["init"],

    random_state=TSNE_CONFIG["random_state"],

)

X_tsne = tsne.fit_transform(

    X_tsne_input

)

# ==============================================================================
# DataFrame
# ==============================================================================

tsne_df = pd.DataFrame({

    "TSNE_1": X_tsne[:,0],

    "TSNE_2": X_tsne[:,1],

    "Label": label_encoder.inverse_transform(y_tsne),

})

# Cache the sample count before freeing -- the summary table below
# still needs it, but X_tsne_input itself is fully represented by
# tsne_df now.
TSNE_SAMPLES_USED = len(X_tsne_input)

free_memory("X_tsne", "X_tsne_input", "y_tsne", scope=globals())

# ==============================================================================
# Plot
# ==============================================================================

plt.figure(figsize=(12,8))

sns.scatterplot(

    data=tsne_df,

    x="TSNE_1",

    y="TSNE_2",

    hue="Label",

    s=12,

    alpha=0.7,

)

plt.title(

    "t-SNE Visualization",

    fontsize=18,

)

plt.legend(

    bbox_to_anchor=(1.02,1),

    loc="upper left",

)

plt.tight_layout()

plt.savefig(

    FIGURE_DIR / "tsne_visualization.png",

    dpi=300,

)

plt.show()

# ==============================================================================
# Save
# ==============================================================================

tsne_df.to_csv(

    PROCESSED_DATASET_DIR / "tsne_dataset.csv",

    index=False,

)

joblib.dump(

    tsne,

    MODEL_DIR / "tsne.pkl",

)

summary = pd.DataFrame({

    "Metric":[

        "Dataset Used",

        "Samples",

        "Perplexity",

        "Learning Rate",

        "Initialization",

    ],

    "Value":[

        "Full Dataset"

        if TSNE_CONFIG["use_full_dataset"]

        else "Sample Dataset",

        TSNE_SAMPLES_USED,

        TSNE_CONFIG["perplexity"],

        TSNE_CONFIG["learning_rate"],

        TSNE_CONFIG["init"],

    ]

})

display(summary)

summary.to_csv(

    TABLE_DIR / "tsne_summary.csv",

    index=False,

)

print()

print("t-SNE Completed Successfully.")

print("=" * 80)
gc.collect()
print_ram_usage("after t-SNE")


In [ ]:
# ==============================================================================
# Cell 16 : UMAP Visualization
# ==============================================================================

print("=" * 80)
print("UMAP Visualization")
print("=" * 80)

# ==============================================================================
# Install UMAP if Needed
# ==============================================================================

try:

    import umap.umap_ as umap

except ImportError:

    print("Installing UMAP...")

    import sys
    import subprocess

    subprocess.check_call(

        [sys.executable, "-m", "pip", "install", "-q", "umap-learn"]

    )

    import umap.umap_ as umap

# ==============================================================================
# Configuration
# ==============================================================================

# True  -> use entire dataset
# False -> sample dataset
use_full_dataset = False        #@param {type:"boolean"}

# Number of samples if use_full_dataset=False
sample_size = 10000       #@param {type:"integer"}

# Recommended: True
stratified_sampling = True       #@param {type:"boolean"}

UMAP_CONFIG = {

    # True -> Use Full Dataset
    # False -> Use Sample
    "use_full_dataset": use_full_dataset,

    "sample_size": sample_size,

    "stratified_sampling": stratified_sampling,

    "n_neighbors": 15,

    "min_dist": 0.1,

    "metric": "euclidean",

    "random_state": SEED,

}

# ==============================================================================
# Dataset Selection
# ==============================================================================

if UMAP_CONFIG["use_full_dataset"]:

    print("\nUsing FULL dataset.")

    X_umap_input = X_train_scaled.copy()

    y_umap = y_train_smote.copy()

else:

    print("\nUsing SAMPLE dataset.")

    sample_size = min(

        UMAP_CONFIG["sample_size"],

        len(X_train_scaled)

    )

    if UMAP_CONFIG["stratified_sampling"]:

        _, X_umap_input, _, y_umap = train_test_split(

            X_train_scaled,

            y_train_smote,

            test_size=sample_size,

            stratify=y_train_smote,

            random_state=SEED,

        )

    else:

        indices = np.random.choice(

            len(X_train_scaled),

            sample_size,

            replace=False,

        )

        X_umap_input = X_train_scaled.iloc[indices]

        y_umap = y_train_smote[indices]

print(f"Samples Used : {len(X_umap_input):,}")

# ==============================================================================
# Run UMAP
# ==============================================================================

print("\nRunning UMAP...")

reducer = umap.UMAP(

    n_neighbors=UMAP_CONFIG["n_neighbors"],

    min_dist=UMAP_CONFIG["min_dist"],

    metric=UMAP_CONFIG["metric"],

    random_state=UMAP_CONFIG["random_state"],

)

embedding = reducer.fit_transform(

    X_umap_input

)

# ==============================================================================
# Create DataFrame
# ==============================================================================

umap_df = pd.DataFrame({

    "UMAP_1": embedding[:,0],

    "UMAP_2": embedding[:,1],

    "Label": label_encoder.inverse_transform(y_umap),

})

# Cache the sample count before freeing -- the summary table below
# still needs it, but X_umap_input itself is fully represented by
# umap_df now.
UMAP_SAMPLES_USED = len(X_umap_input)

free_memory("embedding", "X_umap_input", "y_umap", scope=globals())

# ==============================================================================
# Visualization
# ==============================================================================

plt.figure(figsize=(12,8))

sns.scatterplot(

    data=umap_df,

    x="UMAP_1",

    y="UMAP_2",

    hue="Label",

    s=12,

    alpha=0.7,

)

plt.title(

    "UMAP Projection",

    fontsize=18,

)

plt.legend(

    bbox_to_anchor=(1.02,1),

    loc="upper left",

)

plt.tight_layout()

plt.savefig(

    FIGURE_DIR / "umap_visualization.png",

    dpi=300,

)

plt.show()

# ==============================================================================
# Save Results
# ==============================================================================

umap_df.to_csv(

    PROCESSED_DATASET_DIR / "umap_dataset.csv",

    index=False,

)

joblib.dump(

    reducer,

    MODEL_DIR / "umap.pkl",

)

summary = pd.DataFrame({

    "Metric":[

        "Dataset Used",

        "Samples",

        "Neighbors",

        "Minimum Distance",

        "Metric",

    ],

    "Value":[

        "Full Dataset"

        if UMAP_CONFIG["use_full_dataset"]

        else "Sample Dataset",

        UMAP_SAMPLES_USED,

        UMAP_CONFIG["n_neighbors"],

        UMAP_CONFIG["min_dist"],

        UMAP_CONFIG["metric"],

    ]

})

display(summary)

summary.to_csv(

    TABLE_DIR / "umap_summary.csv",

    index=False,

)

print()

print("UMAP Completed Successfully.")

print("=" * 80)
gc.collect()
print_ram_usage("after UMAP")


In [ ]:
# ==============================================================================
# Cell 17 : Final Dataset Export
# ==============================================================================

print("=" * 80)
print("Exporting Final Preprocessed Dataset")
print("=" * 80)

import shutil

# ==============================================================================
# Export Directory
# ==============================================================================

EXPORT_DIR = PROCESSED_DATASET_DIR / "final"

EXPORT_DIR.mkdir(

    parents=True,

    exist_ok=True,

)

# ==============================================================================
# Preflight Disk-Space Check
# ==============================================================================

largest_frame_bytes = max(

    X_train_smote.memory_usage(deep=True).sum(),

    X_train_scaled.memory_usage(deep=True).sum(),

)

free_bytes = shutil.disk_usage(EXPORT_DIR).free

required_bytes = largest_frame_bytes * 2

print(f"Free Disk Space   : {free_bytes / (1024**3):.2f} GB")

print(f"Estimated Needed  : {required_bytes / (1024**3):.2f} GB")

if free_bytes < required_bytes:

    raise RuntimeError(

        f"Insufficient disk space for export. "

        f"Free: {free_bytes / (1024**3):.2f} GB, "

        f"Estimated needed: {required_bytes / (1024**3):.2f} GB. "

        f"Free up space or lower CONFIG[\'smote_target_per_class\']."

    )

print("Disk space check passed.\n")

print_ram_usage("before export")

# ==============================================================================
# Save Large Frames One at a Time (Parquet, Snappy)
# ==============================================================================
# NOTE: previously all four large frames (X_train_smote, X_train_scaled,
# X_validation_scaled, X_test_scaled) were held in a dict simultaneously
# just to loop over them -- that dict doesn't add real memory since it
# only holds references, but writing them back-to-back with a gc.collect()
# between each write ensures pyarrow's internal buffers from writing one
# file are released before the next (large) write starts, instead of
# stacking up.

large_datasets = {

    "X_train_smote.parquet": "X_train_smote",

    "X_train_scaled.parquet": "X_train_scaled",

    "X_validation_scaled.parquet": "X_validation_scaled",

    "X_test_scaled.parquet": "X_test_scaled",

}

for filename, varname in large_datasets.items():

    globals()[varname].to_parquet(

        EXPORT_DIR / filename,

        engine="pyarrow",

        compression="snappy",

        index=False,

    )

    gc.collect()

    print(f"Saved {filename}")

# ==============================================================================
# Save Smaller Reference Frames as CSV
# ==============================================================================
# NOTE: dropped "encoded_dataset.csv" from this export -- that variable
# doesn't exist any more (Cell 7 stopped building it on purpose to avoid
# a redundant full-size copy of X). X_train/X_val/X_test here are the
# PRE-SCALING, PRE-SMOTE splits, kept only as a small CSV reference;
# the versions actually used for modeling are the *_scaled.parquet /
# X_train_smote.parquet files saved above.

small_datasets = {

    "X_train.csv": X_train,

    "X_validation.csv": X_val,

    "X_test.csv": X_test,

}

for filename, dataframe in small_datasets.items():

    dataframe.to_csv(

        EXPORT_DIR / filename,

        index=False,

    )

# selected_features.parquet was already written to PROCESSED_DATASET_DIR
# in Cell 9 -- copy it into the export folder instead of re-writing
# X_selected, which no longer exists in memory at this point anyway
# (it was freed right after the train/val/test split in Cell 11).

src_selected = PROCESSED_DATASET_DIR / "selected_features.parquet"

if src_selected.exists():

    shutil.copy2(

        src_selected,

        EXPORT_DIR / "selected_features.parquet",

    )

# ==============================================================================
# Save Labels
# ==============================================================================
# NOTE: y_train_smote.parquet corresponds row-for-row to
# X_train_smote.parquet / X_train_scaled.parquet (both already
# SMOTE-balanced). y_train.csv corresponds to the ORIGINAL pre-SMOTE
# X_train.csv. Do not mix a *_smote frame with the plain y_train.csv.
# When loading X_train_scaled downstream, set USE_SMOTE_PER_FOLD=False
# since it is already SMOTE-balanced.

pd.DataFrame({

    "Label": y_train

}).to_csv(

    EXPORT_DIR / "y_train.csv",

    index=False,

)

pd.DataFrame({

    "Label": y_val

}).to_csv(

    EXPORT_DIR / "y_validation.csv",

    index=False,

)

pd.DataFrame({

    "Label": y_test

}).to_csv(

    EXPORT_DIR / "y_test.csv",

    index=False,

)

pd.DataFrame({

    "Label": y_train_smote

}).to_parquet(

    EXPORT_DIR / "y_train_smote.parquet",

    engine="pyarrow",

    compression="snappy",

    index=False,

)

# ==============================================================================
# Save NumPy Arrays (Written Directly to Disk, One at a Time)
# ==============================================================================
# NOTE: np.save(..., X_train_scaled.values) materializes a full second
# copy of X_train_scaled as a raw ndarray right before writing. Freeing
# X_train_scaled/X_train_smote AFTER this block (not before) since the
# .npy files still need them, but doing the four saves sequentially
# with gc.collect() between them avoids holding multiple .values
# copies in memory at the same time.

np.save(EXPORT_DIR / "X_train.npy", X_train_scaled.values)
gc.collect()

np.save(EXPORT_DIR / "X_validation.npy", X_validation_scaled.values)
gc.collect()

np.save(EXPORT_DIR / "X_test.npy", X_test_scaled.values)
gc.collect()

np.save(EXPORT_DIR / "y_train.npy", np.array(y_train_smote))

np.save(EXPORT_DIR / "y_validation.npy", np.array(y_val))

np.save(EXPORT_DIR / "y_test.npy", np.array(y_test))

gc.collect()

print("\nAll .npy arrays saved.")

print_ram_usage("after npy export")

# ==============================================================================
# Save Metadata
# ==============================================================================

metadata = {

    "created": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),

    "random_seed": SEED,

    "selected_features": len(selected_features) if "selected_features" in globals() else None,

    "number_of_classes": int(len(label_encoder.classes_)),

    "class_names": label_encoder.classes_.tolist(),

    "scaler": "StandardScaler",

    "feature_selection": "Random Forest" if CONFIG["use_feature_selection"] else "None (all features kept)",
    "feature_selection_enabled": CONFIG["use_feature_selection"],

    "smote": True,

    "smote_target_per_class": CONFIG["smote_target_per_class"],

    "smote_output_format": "parquet (snappy)",

    "use_smote_per_fold": False,

}

with open(EXPORT_DIR / "metadata.json", "w") as f:

    json.dump(metadata, f, indent=4)

# ==============================================================================
# Manifest
# ==============================================================================

manifest = pd.DataFrame({

    "File": [f.name for f in sorted(EXPORT_DIR.iterdir())]

})

manifest.to_csv(EXPORT_DIR / "manifest.csv", index=False)

# ==============================================================================
# Summary
# ==============================================================================

summary = pd.DataFrame({

    "Metric": ["Export Folder", "Files Exported"],

    "Value": [str(EXPORT_DIR), len(list(EXPORT_DIR.iterdir()))],

})

display(summary)

summary.to_csv(TABLE_DIR / "export_summary.csv", index=False)

# ==============================================================================
# Cache Pipeline Stats (for Cell 18 / 19 -- read AFTER memory wipe below)
# ==============================================================================
# From this point on, the big DataFrames get freed to keep RAM low for
# the rest of the notebook. Cells 18/19 need a few summary numbers, so
# we capture them here, while everything is still alive, and write
# them to disk. Downstream cells read this file instead of touching
# the (by-then-freed) DataFrames directly.

PIPELINE_STATS = {

    "dataset_rows": int(DATASET_SHAPE[0]),

    "dataset_columns": int(DATASET_SHAPE[1]),

    "selected_features": int(X_selected.shape[1]) if "X_selected" in globals() else len(feature_names),

    "training_samples": int(len(X_train_smote)),

    "validation_samples": int(len(X_val)),

    "testing_samples": int(len(X_test)),

    "classes": int(len(label_encoder.classes_)),

    "smote_target_per_class": CONFIG["smote_target_per_class"],

    "export_dir": str(EXPORT_DIR),

}

with open(

    PREPROCESS_REPORT_DIR / "pipeline_stats.json",

    "w",

) as f:

    json.dump(

        PIPELINE_STATS,

        f,

        indent=4,

    )

print()

print("Pipeline stats cached to:")

print(PREPROCESS_REPORT_DIR / "pipeline_stats.json")

# ==============================================================================
# Full Memory Wipe
# ==============================================================================
# Everything needed downstream (Cell 18 report, Cell 19 smoke test) is
# now either on disk (Parquet/CSV/NPY under EXPORT_DIR) or cached in
# PIPELINE_STATS above. None of the big in-memory DataFrames/arrays
# below are touched again, so free them now rather than carrying them
# for the rest of the run -- this is usually the difference between
# finishing cleanly and the Colab runtime crashing near the end.

free_memory(

    "X_train", "X_val", "X_test",

    "X_train_smote", "y_train_smote",

    "X_train_scaled", "X_validation_scaled", "X_test_scaled",

    "y_train", "y_val", "y_test",

    "X_selected", "encoded_dataset",

    "large_datasets", "small_datasets",

    scope=globals(),

)

gc.collect()

print_ram_usage("after full memory wipe (post-export)")

print()

print("All datasets exported successfully.")

print(EXPORT_DIR)

print("=" * 80)


In [ ]:
# ==============================================================================
# Cell 18 : Final Preprocessing Report
# ==============================================================================
# NOTE: This cell deliberately does NOT touch `dataset`, `X_selected`,
# `X_train_smote`, etc. -- those were freed at the end of Cell 17 to
# keep RAM low. Everything here is read back from PIPELINE_STATS
# (cached in Cell 17) or discovered directly from files on disk.

print("=" * 80)
print("FlowGraphFormer - Final Preprocessing Report")
print("=" * 80)

from datetime import datetime
import json

# ==============================================================================
# Load Cached Pipeline Stats
# ==============================================================================

if "PIPELINE_STATS" not in globals():

    with open(

        PREPROCESS_REPORT_DIR / "pipeline_stats.json",

    ) as f:

        PIPELINE_STATS = json.load(f)

# ==============================================================================
# Count Files
# ==============================================================================

figure_files = list(FIGURE_DIR.rglob("*.*"))
table_files = list(TABLE_DIR.rglob("*.*"))
processed_files = list(PROCESSED_DATASET_DIR.rglob("*.*"))
model_files = list(MODEL_DIR.rglob("*.*"))

# ==============================================================================
# Report Dictionary
# ==============================================================================

report = {

    "generated_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),

    "dataset": {

        "samples": PIPELINE_STATS["dataset_rows"],

        "features": PIPELINE_STATS["dataset_columns"],

        "selected_features": PIPELINE_STATS["selected_features"],

        "training_samples": PIPELINE_STATS["training_samples"],

        "validation_samples": PIPELINE_STATS["validation_samples"],

        "testing_samples": PIPELINE_STATS["testing_samples"],

        "classes": PIPELINE_STATS["classes"],

    },

    "preprocessing": {

        "feature_selection": PIPELINE_STATS.get("feature_selection", "Random Forest"),

        "smote": True,

        "smote_target_per_class": PIPELINE_STATS["smote_target_per_class"],

        "scaler": "StandardScaler",

        "pca": True,

        "tsne": True,

        "umap": True,

    },

    "generated_files": {

        "figures": len(figure_files),

        "tables": len(table_files),

        "processed_files": len(processed_files),

        "saved_models": len(model_files),

    },

}

# ==============================================================================
# Summary Table
# ==============================================================================

summary = pd.DataFrame({

    "Category":[

        "Final Samples",

        "Final Features",

        "Selected Features",

        "Training Samples",

        "Validation Samples",

        "Testing Samples",

        "Classes",

        "Figures",

        "Tables",

        "Processed Files",

        "Saved Models",

    ],

    "Value":[

        PIPELINE_STATS["dataset_rows"],

        PIPELINE_STATS["dataset_columns"],

        PIPELINE_STATS["selected_features"],

        PIPELINE_STATS["training_samples"],

        PIPELINE_STATS["validation_samples"],

        PIPELINE_STATS["testing_samples"],

        PIPELINE_STATS["classes"],

        len(figure_files),

        len(table_files),

        len(processed_files),

        len(model_files),

    ]

})

display(summary)

# ==============================================================================
# Save Report
# ==============================================================================

summary.to_csv(

    PREPROCESS_REPORT_DIR / "preprocessing_report.csv",

    index=False,

)

with open(

    PREPROCESS_REPORT_DIR / "preprocessing_report.json",

    "w",

) as f:

    json.dump(

        report,

        f,

        indent=4,

    )

# ==============================================================================
# Print Generated Files
# ==============================================================================

print("\n")
print("=" * 80)
print("Generated Figures")
print("=" * 80)

for file in sorted(figure_files):

    print(file.name)

print("\n")
print("=" * 80)
print("Generated Tables")
print("=" * 80)

for file in sorted(table_files):

    print(file.name)

print("\n")
print("=" * 80)
print("Processed Dataset Files")
print("=" * 80)

for file in sorted(processed_files):

    if file.is_file():

        print(file.relative_to(PROJECT_ROOT))

print("\n")
print("=" * 80)
print("Saved Models / Objects")
print("=" * 80)

for file in sorted(model_files):

    if file.is_file():

        print(file.name)

print("\n")
print("=" * 80)
print("PREPROCESSING COMPLETED SUCCESSFULLY")
print("=" * 80)

print("\nReports Saved:")
print(PREPROCESS_REPORT_DIR / "preprocessing_report.csv")
print(PREPROCESS_REPORT_DIR / "preprocessing_report.json")
print("\nReady for 03_Big_Modeling.ipynb")
print("=" * 80)

gc.collect()
print_ram_usage("after final report")


In [ ]:
# ==============================================================================
# Cell 19 : Preprocessing Validation (Smoke Test)
# ==============================================================================
# NOTE: In-memory DataFrames (X_train_smote, X_train_scaled, etc.) were
# freed at the end of Cell 17. This cell validates the PIPELINE from
# disk instead of from those variables: file existence for structural
# checks, and a small row-limited read (via pyarrow, not pandas) for
# the NaN check, so we never reload a multi-hundred-thousand-row frame
# into RAM just to run a smoke test.

print("=" * 80)
print("FlowGraphFormer Preprocessing Validation")
print("=" * 80)

import pyarrow.parquet as pq

output_dir = PREPROCESS_REPORT_DIR / "tests"
output_dir.mkdir(parents=True, exist_ok=True)

if "PIPELINE_STATS" not in globals():

    with open(PREPROCESS_REPORT_DIR / "pipeline_stats.json") as f:

        PIPELINE_STATS = json.load(f)

EXPORT_DIR = Path(PIPELINE_STATS["export_dir"])

results = {}

# ==============================================================================
# Helper
# ==============================================================================

def check(name, condition):

    results[name] = bool(condition)

    status = "PASS" if condition else "FAIL"

    print(f"{name:.<45}{status}")


def sample_has_nan(parquet_path, n_rows=2000):
    """Read a small row-limited sample of a parquet file and check for NaNs,
    without loading the full file into memory."""

    table = pq.ParquetFile(parquet_path).read_row_group(0) if pq.ParquetFile(parquet_path).num_row_groups > 0 else pq.ParquetFile(parquet_path).read()

    sample_df = table.to_pandas().head(n_rows)

    has_nan = bool(sample_df.isnull().values.any())

    del table, sample_df

    return has_nan

# ==============================================================================
# Pipeline Stats Sanity Checks
# ==============================================================================

check(
    "Pipeline Stats Cached",
    PIPELINE_STATS.get("training_samples", 0) > 0
)

check(
    "Label Encoder Classes Recorded",
    PIPELINE_STATS.get("classes", 0) > 0
)

# ==============================================================================
# Exported Data Files (Parquet -- SMOTE / Scaled)
# ==============================================================================

required_parquet_files = [

    EXPORT_DIR / "X_train_smote.parquet",

    EXPORT_DIR / "X_train_scaled.parquet",

    EXPORT_DIR / "X_validation_scaled.parquet",

    EXPORT_DIR / "X_test_scaled.parquet",

    EXPORT_DIR / "y_train_smote.parquet",

]

for file in required_parquet_files:

    check(
        f"File : {file.name}",
        file.exists()
    )

# ==============================================================================
# Exported Data Files (CSV -- Reference / Pre-SMOTE)
# ==============================================================================

required_csv_files = [

    EXPORT_DIR / "X_train.csv",

    EXPORT_DIR / "X_validation.csv",

    EXPORT_DIR / "X_test.csv",

    EXPORT_DIR / "y_train.csv",

    EXPORT_DIR / "y_validation.csv",

    EXPORT_DIR / "y_test.csv",

    EXPORT_DIR / "metadata.json",

]

for file in required_csv_files:

    check(
        f"File : {file.name}",
        file.exists()
    )

# ==============================================================================
# NaN Check (Row-Limited Sample -- Not a Full Reload)
# ==============================================================================

for file in [

    EXPORT_DIR / "X_train_scaled.parquet",

    EXPORT_DIR / "X_validation_scaled.parquet",

    EXPORT_DIR / "X_test_scaled.parquet",

]:

    if file.exists():

        check(
            f"No NaN (sample) : {file.name}",
            not sample_has_nan(file)
        )

    else:

        check(f"No NaN (sample) : {file.name}", False)

gc.collect()

# ==============================================================================
# Saved Models
# ==============================================================================

required_models = [

    MODEL_DIR / "standard_scaler.pkl",

    ENCODER_DIR / "label_encoder.pkl",

    MODEL_DIR / "smote.pkl",

    MODEL_DIR / "pca.pkl",

    MODEL_DIR / "umap.pkl",

]

for file in required_models:

    check(
        f"Model : {file.name}",
        file.exists()
    )

# ==============================================================================
# Figures
# ==============================================================================

required_figures = [

    "feature_importance.png",

    "pca_visualization.png",

    "tsne_visualization.png",

    "umap_visualization.png",

]

for figure in required_figures:

    check(
        f"Figure : {figure}",
        (FIGURE_DIR / figure).exists()
    )

# ==============================================================================
# Summary
# ==============================================================================

passed = sum(results.values())
failed = len(results) - passed

print("\n" + "=" * 80)
print(f"Total Tests : {len(results)}")
print(f"Passed      : {passed}")
print(f"Failed      : {failed}")
print("=" * 80)

if failed == 0:

    print("ALL PREPROCESSING TESTS PASSED")

else:

    print("SOME TESTS FAILED")

print("=" * 80)

# ==============================================================================
# Save Results
# ==============================================================================

results_df = pd.DataFrame({

    "Test": list(results.keys()),

    "Status": [

        "PASS" if value else "FAIL"

        for value in results.values()

    ]

})

results_df.to_csv(

    output_dir / "preprocessing_validation_results.csv",

    index=False,

)

with open(

    output_dir / "preprocessing_validation_results.json",

    "w",

) as f:

    json.dump(

        results,

        f,

        indent=4,

    )

print()
print("Saved:")
print(output_dir / "preprocessing_validation_results.csv")
print(output_dir / "preprocessing_validation_results.json")

gc.collect()
print_ram_usage("after validation smoke test")
